In [ ]:
import pandas as pd

In [ ]:
import numpy as np

In [ ]:
import itertools

In [ ]:
from collections import Counter

In [ ]:
deployments = pd.read_csv("../data/raw/deployments.csv")

In [ ]:
proposals = pd.read_csv(
    "../data/raw/proposals.csv",
    low_memory=False # Disable low memory mode to avoid DtypeWarning
) # If this fails, almost certainly the CSV files aren't in data/raw/

In [ ]:
deployments["platform"].value_counts() # How many DAOs per platform?

In [ ]:
proposals["platform"].value_counts() # How many proposals per platform?

In [ ]:
deployments["deployment_id"].duplicated().sum() # Is there any duplicate deployment_id?

In [ ]:
pd.crosstab(
    deployments["platform"],
    deployments["estimated_vp"].isna()
) # For each platform, how many deployments have missing 'estimated_vp' values?

In [ ]:
np.isinf(deployments["estimated_vp"]).sum() # From the remaining deployments, how many have infinite 'estimated_vp' values?

In [ ]:
(deployments["estimated_vp"] == 0).sum() # From the remaining deployments, how many have a value of 0 for 'estimated_vp'?

In [ ]:
pd.crosstab(
    proposals["deployment_id"].isna(),
    proposals["platform_deployment_id"].isna()
) # Is 'platform_deployment_id' filled exactly when 'deployment_id' is missing?
# Zeros on the diagonal mean the two columns never overlap: they are complementary.

In [ ]:
missing_id = proposals[proposals["deployment_id"].isna()] # Keep only the proposals without deployment_id

In [ ]:
missing_id.shape

In [ ]:
missing_id["platform"].value_counts() # Which platforms are missing 'deployment_id' in the proposals dataset?

In [ ]:
missing_id["platform_deployment_id"].notna().sum() # Do all of them have the fallback identifier?

In [ ]:
missing_id["platform_deployment_id"].isin(deployments["deployment_id"]).sum() # Are the fallback values also present in the deployments dataset?

In [ ]:
missing_id["platform_deployment_id"].head() # Show the first few fallback values

In [ ]:
deployments[deployments["platform"] == "snapshot"]["deployment_id"].head() # Compare with Snapshot ids in deployments: same *.eth format

In [ ]:
proposals[proposals["total_vp"].isna()]["platform"].value_counts() # Which platforms have missing 'total_vp' values? (same 20,112 Snapshot rows)

In [ ]:
# Create a new column 'dao_id': 'deployment_id' where available, 'platform_deployment_id' where it is missing.
proposals["dao_id"] = proposals["deployment_id"].fillna(
    proposals["platform_deployment_id"]
) # With this column every proposal has an identifier for the DAO it belongs to.

proposals["dao_id"].isna().sum() # Should be 0

In [ ]:
# Proposals whose 'deployment_id' is not found in the deployments dataset.
# At first sight they look like "orphan" proposals, but see the next cells.
orphan_proposals = proposals[
    proposals["deployment_id"].notna()
    & ~proposals["deployment_id"].isin(deployments["deployment_id"])
]

In [ ]:
orphan_proposals.shape # Number of apparent orphan proposals

In [ ]:
orphan_proposals["platform"].value_counts() # Apparent orphans by platform

In [ ]:
deployments[deployments["platform"] == "tally"]["deployment_id"].head() # Tally ids in deployments.csv: all lowercase

In [ ]:
proposals[proposals["platform"] == "tally"]["deployment_id"].head() # Tally ids in proposals.csv: mixed case (checksum format)

In [ ]:
# Ethereum addresses are case-insensitive, so the same DAO is written in two ways.
# Compare both sides in lowercase to see how many apparent orphans are just this.
deployment_ids_lower = set(deployments["deployment_id"].str.lower())
recovered = orphan_proposals["deployment_id"].str.lower().isin(deployment_ids_lower)

In [ ]:
recovered.sum().item() # type: ignore  # Apparent orphans that match once lowercased (all the Tally ones)

In [ ]:
orphan_proposals[~recovered]["platform"].value_counts() # Real orphans: DAO genuinely absent from deployments.csv

In [ ]:
proposals[proposals["author"].isna()]["platform"].value_counts() # Which platforms have missing 'author' values? (all of Tally and Realms)

In [ ]:
proposals["author"].nunique() # Number of distinct authors

In [ ]:
# Identify duplicated proposal_id (keep=False marks every repetition, not only the second one)
duplicated_proposals = proposals[
    proposals["proposal_id"].duplicated(keep=False)
]

In [ ]:
duplicated_proposals.shape

In [ ]:
duplicated_proposals["platform"].value_counts() # All from Tally

In [ ]:
proposals.duplicated().sum() # Rows identical in every column

In [ ]:
# Describe the statistics for the deployment metrics (the 50% row is the median)
deployments[["proposals_count", "unique_voters", "votes_count"]].describe()

In [ ]:
# Count deployments with no unique voters
(deployments["unique_voters"] == 0).sum().item() # type: ignore

In [ ]:
# Count deployments with at least 10 unique voters
(deployments["unique_voters"] >= 10).sum().item() # type: ignore

In [ ]:
# Count deployments with at least 100 unique voters
(deployments["unique_voters"] >= 100).sum().item() # type: ignore

In [ ]:
dates = pd.to_datetime(proposals["date"], errors="coerce") # Text to date; unreadable values become NaT instead of raising an error

In [ ]:
dates.min(), dates.max() # Date range covered by the proposals

In [ ]:
# Count the number of missing dates in the proposals dataset
dates.isna().sum().item() # type: ignore

In [ ]:
linked = proposals.dropna(subset=["dao_id", "author"]) # Drop proposals with missing 'dao_id' or 'author'

In [ ]:
daos_per_author = linked.groupby("author")["dao_id"].nunique() # Count the number of distinct DAOs each author has submitted proposals to

In [ ]:
# Number of authors who have submitted proposals to at least 2 distinct DAOs
# Only these authors can create a connection between two DAOs.
(daos_per_author >= 2).sum().item() # type: ignore

In [ ]:
multi_dao_authors = daos_per_author[daos_per_author >= 2].index # Get the authors who have submitted proposals to at least 2 distinct DAOs

In [ ]:
multi = linked[linked["author"].isin(multi_dao_authors)] # Keep only the proposals from those authors

In [ ]:
# Build the edges: for each author, every pair of DAOs they proposed in is one connection.
# sorted() makes A-B and B-A the same pair. The counter adds 1 per shared author,
# so the final value of each pair is the edge weight.
edges = Counter()

for daos in multi.groupby("author")["dao_id"].unique():
    for a, b in itertools.combinations(sorted(daos), 2):
        edges[(a, b)] += 1

len(edges) # Number of edges

In [ ]:
nodes = set(itertools.chain.from_iterable(edges)) # Flatten all pairs into one list of DAOs and remove repetitions

In [ ]:
len(nodes) # Number of DAOs with at least one connection

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
from src.data_loader import load_deployments
from src.data_cleaner import clean_deployments

deployments = load_deployments()
deployments_clean = clean_deployments(deployments)

deployments_clean.head()

,deployment_id,platform,name,proposals_count,unique_voters,votes_count,estimated_vp,website,additional
0,0x0017a25385931ee690dc09df6cca919d1cd007b2,aragon,NaN,2,2.0,2,1.000000e+37,https://etherscan.io/address/0x0017a25385931ee...,NaN
1,0x001cd74c9a99b6c68e93fe69595124407b37aa8e,aragon,shilldao,17,216.0,216,0.000000e+00,https://etherscan.io/address/0x001cd74c9a99b6c...,NaN
2,0x0068990f8910e502dd41eb71fbc506936ca100af,aragon,NaN,0,0.0,0,0.000000e+00,https://etherscan.io/address/0x0068990f8910e50...,NaN
3,0x00b424842685f69912cfd93af7293602c973bf56,aragon,NaN,0,0.0,0,0.000000e+00,https://etherscan.io/address/0x00b424842685f69...,NaN
4,0x00ccb72bd1f72882d89774d9acccce616ed9227d,aragon,NaN,25,16.0,16,0.000000e+00,https://etherscan.io/address/0x00ccb72bd1f7288...,NaN


In [3]:
print("Filas originales:", len(deployments))
print("Filas limpias:", len(deployments_clean))

print("IDs con mayúsculas antes:",
      deployments["deployment_id"].str.contains(r"[A-Z]", na=False).sum())

print("IDs con mayúsculas después:",
      deployments_clean["deployment_id"].str.contains(r"[A-Z]", na=False).sum())

Filas originales: 86489
Filas limpias: 86489
IDs con mayúsculas antes: 0
IDs con mayúsculas después: 0


In [4]:
from src.data_loader import load_proposals
from src.data_cleaner import clean_proposals

proposals = load_proposals()
proposals_clean = clean_proposals(proposals)

print("Filas originales:", len(proposals))
print("Filas limpias:", len(proposals_clean))

print("IDs con mayúsculas antes:",
      proposals["deployment_id"].str.contains(r"[A-Z]", na=False).sum())

print("IDs con mayúsculas después:",
      proposals_clean["deployment_id"].str.contains(r"[A-Z]", na=False).sum())

c:\Users\adric\OneDrive\Desktop\Mapping-Blockchain-Ecosystem\src\data_loader.py:27: DtypeWarning: Columns (7: platform_deployment_id) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(file_path)


Filas originales: 370133
Filas limpias: 370133
IDs con mayúsculas antes: 37635
IDs con mayúsculas después: 0


In [3]:
from src.data_loader import load_proposals
from src.data_cleaner import clean_proposals

proposals = load_proposals()
proposals_clean = clean_proposals(proposals)

print("deployment_id vacíos antes:",
      proposals["deployment_id"].isna().sum())

print("dao_id vacíos después:",
      proposals_clean["dao_id"].isna().sum())

c:\Users\adric\OneDrive\Desktop\Mapping-Blockchain-Ecosystem\src\data_loader.py:27: DtypeWarning: Columns (7: platform_deployment_id) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(file_path)


deployment_id vacíos antes: 20112
dao_id vacíos después: 0


In [5]:
print("Filas totales:", len(proposals))
print("Filas duplicadas exactas:", proposals.duplicated().sum())

Filas totales: 370133
Filas duplicadas exactas: 5823


In [2]:
from src.data_loader import load_proposals
from src.data_cleaner import clean_proposals

proposals = load_proposals()
proposals_clean = clean_proposals(proposals)

print("Filas antes:", len(proposals))
print("Filas después:", len(proposals_clean))
print("Duplicados exactos después:", proposals_clean.duplicated().sum())

c:\Users\adric\OneDrive\Desktop\Mapping-Blockchain-Ecosystem\src\data_loader.py:27: DtypeWarning: Columns (7: platform_deployment_id) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(file_path)


Filas antes: 370133
Filas después: 364310
Duplicados exactos después: 0


In [2]:
from src.data_cleaner import normalize_voter

print("Snapshot:",
      normalize_voter("0xABC123", "snapshot"))

print("Tally:",
      normalize_voter("eip155:1:0xABC123", "tally"))

print("Aragon:",
      normalize_voter("dao123-voter-0xABC123", "aragon"))

print("DAOhaus:",
      normalize_voter("0xABC123", "daohaus"))

print("DAOstack:",
      normalize_voter("0xABC123", "daostack"))

print("Realms:",
      normalize_voter("AbC123XYZ", "realms"))

Snapshot: 0xabc123
Tally: 0xabc123
Aragon: 0xabc123
DAOhaus: 0xabc123
DAOstack: 0xabc123
Realms: AbC123XYZ


In [3]:
from src.data_loader import load_votes
from src.data_cleaner import clean_votes

votes_generator = load_votes()
votes_chunk = next(votes_generator)

votes_clean = clean_votes(votes_chunk)

print("Filas del chunk:", len(votes_chunk))

votes_clean[["platform", "deployment_id", "voter"]].head(10)

C:\Users\adric\AppData\Local\Temp\ipykernel_25716\2772460742.py:5: DtypeWarning: Columns (4: date, 7: weight) have mixed types. Specify dtype option on import or set low_memory=False.
  votes_chunk = next(votes_generator)


Filas del chunk: 1000000


,platform,deployment_id,voter
0,aragon,0x79cf3e23eded0fc20992b79acb3b3ffb7c6799b4,0x0000008cc20c51c19b3557bf95481b6073b9b4e5
1,aragon,0xf4dacf2edb6e2bd88eb90c3aef1e510153d38579,0x000000de5f9e90ce604da5fd78acd6fae789ecca
2,aragon,0xc1a90f60e32321c3c12c6996d57608cfa439641e,0x00000b43329dc00f7eb64b8f9ce210a4b8e2b2a3
3,aragon,0xf95aeaadc21d0a0ad51f29a01ecb226453210cd0,0x002b3d4527a5d7799cc8714733278b675ec36583
4,aragon,0x5ad614d836912ee578636d18edb01f2af2c74846,0x0030026755d09ee4b809b7e968dfe1b004b16286
5,aragon,0x496d97e12f53c899c30bea7f3f4136c87423dbeb,0x0032aaac49a477542d5686d38206236f7de952fa
6,aragon,0x4a46183a9e96a6f58966ada6329beaddaf6cddf0,0x003cc017e32ca02e9657baf5a3a8c30a21700fd0
7,aragon,0x2de83b50af29678774d5abc4a7cb2a588762f28c,0x003dc32fe920a4aaeed12dc87e145f030aa753f3
8,aragon,0xa092ef957538cd20e08a558aa1884e01a0529670,0x0053e4814aadbd8b2fcf86b4b2aac9d8c909ef81
9,aragon,0x054086d40cf8fd5bf6200eda7f9c6877b0302dd1,0x005c25473bf2367aa821a7072332820ac9902547


In [4]:
print("Ejemplos ANTES:")
print(votes_chunk[["platform", "voter"]].head(5))

print("\nEjemplos DESPUÉS:")
print(votes_clean[["platform", "voter"]].head(5))

print("\nVoters modificados:",
      (votes_chunk["voter"].astype("string") !=
       votes_clean["voter"].astype("string")).sum())

Ejemplos ANTES:
  platform                                              voter
0   aragon  0x1115c9b3168563354137cdc60efb66552dd50678-vot...
1   aragon  0x9b8e397c483449623525efda8f80d9b52481a3a1-vot...
2   aragon  0xc50f36e3db27d3f4e303384ca85503baa9b02510-vot...
3   aragon  0xa1316e773519bdc6f135509962b51f318869cc25-vot...
4   aragon  0x7bc51cfcc7a538a6cce75d42285502937b5f3c2d-vot...

Ejemplos DESPUÉS:
  platform                                       voter
0   aragon  0x0000008cc20c51c19b3557bf95481b6073b9b4e5
1   aragon  0x000000de5f9e90ce604da5fd78acd6fae789ecca
2   aragon  0x00000b43329dc00f7eb64b8f9ce210a4b8e2b2a3
3   aragon  0x002b3d4527a5d7799cc8714733278b675ec36583
4   aragon  0x0030026755d09ee4b809b7e968dfe1b004b16286

Voters modificados: 26833


In [2]:
from src.data_loader import load_deployments, load_proposals
from src.data_cleaner import (
    clean_deployments,
    clean_proposals,
    save_processed_data
)

deployments = load_deployments()
proposals = load_proposals()

deployments_clean = clean_deployments(deployments)
proposals_clean = clean_proposals(proposals)

save_processed_data(deployments_clean, proposals_clean)

print("Datos procesados guardados correctamente")

c:\Users\adric\OneDrive\Desktop\Mapping-Blockchain-Ecosystem\src\data_loader.py:27: DtypeWarning: Columns (7: platform_deployment_id) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(file_path)


Datos procesados guardados correctamente


In [2]:
from src.data_loader import load_votes
from src.data_cleaner import process_votes

votes_generator = load_votes()

process_votes(votes_generator)

c:\Users\adric\OneDrive\Desktop\Mapping-Blockchain-Ecosystem\src\data_cleaner.py:88: DtypeWarning: Columns (4: date, 7: weight) have mixed types. Specify dtype option on import or set low_memory=False.
  for i, chunk in enumerate(votes_generator, start=1):


Chunk 1 procesado - 1000000 filas


c:\Users\adric\OneDrive\Desktop\Mapping-Blockchain-Ecosystem\src\data_cleaner.py:88: DtypeWarning: Columns (4: date, 6: choice) have mixed types. Specify dtype option on import or set low_memory=False.
  for i, chunk in enumerate(votes_generator, start=1):


Chunk 2 procesado - 1000000 filas
Chunk 3 procesado - 1000000 filas
Chunk 4 procesado - 1000000 filas
Chunk 5 procesado - 1000000 filas
Chunk 6 procesado - 1000000 filas
Chunk 7 procesado - 1000000 filas
Chunk 8 procesado - 1000000 filas
Chunk 9 procesado - 1000000 filas


c:\Users\adric\OneDrive\Desktop\Mapping-Blockchain-Ecosystem\src\data_cleaner.py:88: DtypeWarning: Columns (7: weight) have mixed types. Specify dtype option on import or set low_memory=False.
  for i, chunk in enumerate(votes_generator, start=1):


Chunk 10 procesado - 1000000 filas


c:\Users\adric\OneDrive\Desktop\Mapping-Blockchain-Ecosystem\src\data_cleaner.py:88: DtypeWarning: Columns (0: vote_id, 1: proposal_id) have mixed types. Specify dtype option on import or set low_memory=False.
  for i, chunk in enumerate(votes_generator, start=1):


Chunk 11 procesado - 1000000 filas
Chunk 12 procesado - 638348 filas


In [3]:
import pandas as pd
from pathlib import Path

processed = Path("../data/processed")

deployments_check = pd.read_csv(processed / "deployments_clean.csv")
proposals_check = pd.read_csv(processed / "proposals_clean.csv")

print("Deployments:", len(deployments_check))
print("Proposals:", len(proposals_check))

votes_rows = 0

for chunk in pd.read_csv(
    processed / "votes_clean.csv",
    chunksize=1_000_000,
    low_memory=False
):
    votes_rows += len(chunk)

print("Votes:", votes_rows)

C:\Users\adric\AppData\Local\Temp\ipykernel_9844\577085563.py:7: DtypeWarning: Columns (7: platform_deployment_id) have mixed types. Specify dtype option on import or set low_memory=False.
  proposals_check = pd.read_csv(processed / "proposals_clean.csv")


Deployments: 86489
Proposals: 364310
Votes: 11638348


In [4]:
print("=== DEPLOYMENTS ===")
print("Duplicados:", deployments_check.duplicated().sum())
print(
    "IDs con mayúsculas:",
    deployments_check["deployment_id"]
    .astype("string")
    .str.contains(r"[A-Z]", na=False)
    .sum()
)

print("\n=== PROPOSALS ===")
print("Duplicados:", proposals_check.duplicated().sum())
print("dao_id vacíos:", proposals_check["dao_id"].isna().sum())
print(
    "deployment_id con mayúsculas:",
    proposals_check["deployment_id"]
    .astype("string")
    .str.contains(r"[A-Z]", na=False)
    .sum()
)

=== DEPLOYMENTS ===
Duplicados: 0
IDs con mayúsculas: 0

=== PROPOSALS ===
Duplicados: 0
dao_id vacíos: 0
deployment_id con mayúsculas: 0


In [5]:
votes_count = 0
uppercase_deployment_ids = 0
missing_voters = 0

for chunk in pd.read_csv(
    processed / "votes_clean.csv",
    chunksize=1_000_000,
    low_memory=False
):
    votes_count += len(chunk)

    uppercase_deployment_ids += (
        chunk["deployment_id"]
        .astype("string")
        .str.contains(r"[A-Z]", na=False)
        .sum()
    )

    missing_voters += chunk["voter"].isna().sum()

print("=== VOTES ===")
print("Filas:", votes_count)
print("deployment_id con mayúsculas:", uppercase_deployment_ids)
print("voter vacíos:", missing_voters)

=== VOTES ===
Filas: 11638348
deployment_id con mayúsculas: 0
voter vacíos: 0
